<a href="https://colab.research.google.com/github/vivianagonzalez203/Bootcamp-Data-Science/blob/main/02_Python/Clase_08_Taller_Pandas_%26_NumPy_Cafe_Sales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller práctico: Limpieza de datos con Pandas + NumPy


**Dataset:** Dirty Cafe Sales  


## Reto

Una cafetería nos entregó su historial de transacciones, pero la base contiene datos faltantes, valores inválidos, tipos incorrectos y registros duplicados.

Tu misión es:

**diagnosticar → limpiar → transformar → validar**

Al finalizar, queremos conseguir un DataFrame completamente limpio:

```python
df.isna().sum().sum()
```

debe devolver:

```text
0
```

> **Importante:** no vamos a aplicar la misma solución a todas las columnas. Tendrás que decidir qué estrategia corresponde a cada problema.


In [60]:
import pandas as pd
import numpy as np

## 0. Preparación

Vamos a trabajar directamente con el archivo desde una URL, así todos utilizaremos la misma fuente de datos.

Ejecuta la siguiente celda.

In [61]:
url = "https://raw.githubusercontent.com/shaadyalii/Dirty-Cafe-Sales-Dataset/main/dirty_cafe_sales.csv"

df = pd.read_csv(url)

df.head()


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


## 1. Diagnóstico inicial

### Pregunta 1

Cuántas filas y columnas tiene el DataFrame?

In [62]:
df.shape


(10000, 8)

### Pregunta 2

Muestra:

- los nombres de las columnas
- los tipos de datos
- información general del DataFrame

Puedes utilizar `columns` e `info()`.

In [63]:
df.info()



<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [64]:
df.columns

Index(['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent',
       'Payment Method', 'Location', 'Transaction Date'],
      dtype='str')

### Pregunta 3

Cuántos valores nulos tiene cada columna?

Recuerda que `isna()` identifica valores ausentes.

In [65]:
df.isnull().sum()


Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

### Pregunta 4

Cuántos registros duplicados existen?

In [66]:
df.duplicated().sum()


np.int64(0)

## 2. Detectar valores inválidos

Un dato problemático no siempre aparece como `NaN`.

En esta base también podemos encontrar valores como:

```text
ERROR
UNKNOWN
```

### Pregunta 5

Busca en qué columnas aparecen estos valores y cuántos hay.

In [67]:
for col in df.columns:
    error = df[col].isin(["ERROR","UNKNOWN"]).sum()
    if error > 0:
        print(col,error)
# Pista: puedes recorrer las columnas con un for


Item 636
Quantity 341
Price Per Unit 354
Total Spent 329
Payment Method 599
Location 696
Transaction Date 301


### Pregunta 6

Reemplaza `ERROR` y `UNKNOWN` por `np.nan`.


In [68]:
df = df.replace(["ERROR","UNKNOWN"],np.nan)



### Pregunta 7

Después de reemplazar los valores inválidos, vuelve a revisar los valores nulos.

Aumentó el número de valores faltantes? Por qué?

In [69]:
df.isnull().sum()


Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

## 3. Eliminar duplicados

### Pregunta 8

Elimina los registros duplicados.



In [70]:
filas_originales= len (df)
df= df.drop_duplicates()
filas_despues = len(df)
print(filas_originales)
print(filas_despues)
print(filas_originales - filas_despues)

10000
10000
0


## 4. Conversión de tipos

### Pregunta 9

Convierte estas columnas a numéricas:

- `Quantity`
- `Price Per Unit`
- `Total Spent`

**Pista:**

```python
pd.to_numeric()
```

y utiliza `errors="coerce"`.

In [71]:
columnas_numericas = ["Quantity", "Price Per Unit","Total Spent"]
for col in columnas_numericas:
    df[col] = pd.to_numeric(df[col], errors = "coerce")
df[columnas_numericas].dtypes


Quantity          float64
Price Per Unit    float64
Total Spent       float64
dtype: object

### Pregunta 10

Convierte `Transaction Date` a formato fecha.

**Pista:**

```python
pd.to_datetime()
```

In [72]:
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors = "coerce")
df["Transaction Date"].dtypes


dtype('<M8[us]')

### Pregunta 11

Comprueba que las conversiones funcionaron correctamente.

Qué tipo de dato tiene ahora cada columna?

In [73]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  str           
 1   Item              9031 non-null   str           
 2   Quantity          9521 non-null   float64       
 3   Price Per Unit    9467 non-null   float64       
 4   Total Spent       9498 non-null   float64       
 5   Payment Method    6822 non-null   str           
 6   Location          6039 non-null   str           
 7   Transaction Date  9540 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(3), str(4)
memory usage: 625.1 KB


In [74]:
df.describe()

,Quantity,Price Per Unit,Total Spent,Transaction Date
count,9521.000000,9467.000000,9498.000000,9540
mean,3.028463,2.949984,8.924352,2023-07-01 23:00:31.698113
min,1.000000,1.000000,1.000000,2023-01-01 00:00:00
25%,2.000000,2.000000,4.000000,2023-04-01 00:00:00
50%,3.000000,3.000000,8.000000,2023-07-02 00:00:00
75%,4.000000,4.000000,12.000000,2023-10-02 00:00:00
max,5.000000,5.000000,25.000000,2023-12-31 00:00:00
std,1.419007,1.278450,6.009919,NaN


## 5. Tratamiento de valores faltantes

Ahora tenemos que tomar decisiones.

Utilizaremos diferentes estrategias:

| Columna | Estrategia |
|---|---|
| `Item` | `"Unknown"` |
| `Quantity` | Mediana |
| `Price Per Unit` | Mediana |
| `Total Spent` | Recalcular |
| `Payment Method` | Moda |
| `Location` | Moda |
| `Transaction Date` | Fecha mediana |

### Pregunta 12 — `Item`

Tenemos transacciones donde no sabemos qué producto fue vendido.

Sería correcto reemplazar todos los faltantes por el producto más frecuente?



### Pregunta 12

Completa los valores faltantes de `Item` utilizando:

```text
Unknown
```



In [75]:
# Tu código aquí
df["Item"] = df["Item"].fillna("Unknown")
df["Item"].isnull().sum()

np.int64(0)

## 6. NumPy: mediana de `Quantity`

### Pregunta 13

Calcula la mediana de `Quantity` **ignorando los valores nulos**.

Busca una función de NumPy que empiece por:

```python
np.nan...
```

In [76]:
# Tu código aquí
mediana_de_cantidad = np.nanmedian(df["Quantity"])
mediana_de_cantidad

np.float64(3.0)

### Pregunta 14

Utiliza la mediana calculada para completar los valores faltantes de `Quantity`.

Después comprueba que ya no existan valores nulos en esa columna.

In [77]:
# Tu código aquí
df["Quantity"] = df["Quantity"].fillna(mediana_de_cantidad)
df["Quantity"].isnull().sum()

np.int64(0)

## 7. `Price Per Unit`

### Pregunta 15

Calcula la mediana de `Price Per Unit` utilizando NumPy y úsala para completar cualquier valor faltante.


In [78]:
# Tu código aquí
mediana_precio_unitario = np.nanmedian(df["Price Per Unit"])
mediana_precio_unitario

np.float64(3.0)

In [79]:
df["Price Per Unit"] = df["Price Per Unit"].fillna(mediana_precio_unitario)
df["Price Per Unit"].isnull().sum()

np.int64(0)

## 8. Reconstruir `Total Spent`

Esta vez **no vamos a utilizar la mediana**.

Observa las columnas:

```text
Quantity
Price Per Unit
Total Spent
```

Existe una relación entre ellas:

```text
Total Spent = Quantity × Price Per Unit
```

### Pregunta 16

Utiliza esa relación para reconstruir `Total Spent`.



In [80]:
# Tu código aquí
df["Total Spent"] = (df["Quantity"] * df["Price Per Unit"])
df["Total Spent"].isnull().sum()

np.int64(0)

## 9. Variables categóricas

`Payment Method` y `Location` son variables categóricas.

### Pregunta 17

Si todavía existen valores nulos en estas columnas:

1. Calcula la moda.
2. Utiliza la moda para completar los valores faltantes.
3. Comprueba el resultado.

Puedes utilizar un `for` para resolver ambas columnas.

In [81]:
print(df.columns.tolist())

['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date']


In [82]:
# Tu código aquí
for col in ["Payment Method", "Location"]:
    if df[col].isnull().any():
        moda = df[col].mode()[0]
        df[col] = df[col].fillna(moda)
print(df[["Payment Method","Location"]].isnull().sum())      

Payment Method    0
Location          0
dtype: int64


## 10. `Transaction Date`

Tenemos valores faltantes en la fecha de transacción.


### Pregunta 18

Calcula una fecha representativa utilizando la fecha central de los registros disponibles y úsala para completar los valores faltantes.

Pista:

1. Elimina temporalmente los `NaT`.
2. Ordena las fechas.
3. Obtén el valor central.
4. Usa `fillna()`.

In [86]:
# Tu código aquí
mediana_fecha = pd.to_datetime(np.nanmedian(df["Transaction Date"].astype("int64")))


In [89]:
df["Transaction Date"] = df["Transaction Date"].fillna(mediana_fecha)
df["Transaction Date"].isnull().sum()

np.int64(0)

## 11. Crear `Revenue_Calculated`

Ahora que las variables necesarias están limpias, podemos crear una nueva variable.

### Pregunta 19

Calcula:

```text
Revenue = Quantity × Price Per Unit
```

y guarda el resultado en:

```text
Revenue_Calculated
```

In [92]:
# Tu código aquí
df["Revenue_Calculated"] = (df["Quantity"] * df["Price Per Unit"])

### Pregunta 20

Comprueba si `Revenue_Calculated` contiene valores nulos.

In [93]:
# Tu código aquí
df["Revenue_Calculated"].isnull().sum()

np.int64(0)

## 12. `np.where()`

NumPy para crear una clasificación.

### Pregunta 21

Crea una columna llamada `Revenue_Level` con estas reglas:

- `High` si `Revenue_Calculated >= 15`
- `Low` si `Revenue_Calculated < 15`

Utiliza `np.where()`.

La estructura general es:

```python
np.where(condición, valor_si_se_cumple, valor_si_no_se_cumple)
```

In [94]:
# Tu código aquí
df["Revenue_Level"] = np.where(df["Revenue_Calculated"] >= 15, "High", "Low")

## 13. Validación de la limpieza

### Pregunta 22

Cuántos valores nulos quedan en cada columna?

In [95]:
# Tu código aquí
df.isnull().sum()

Transaction ID        0
Item                  0
Quantity              0
Price Per Unit        0
Total Spent           0
Payment Method        0
Location              0
Transaction Date      0
Revenue_Calculated    0
Revenue_Level         0
dtype: int64

In [98]:
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Revenue_Calculated,Revenue_Level
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08,4.0,Low
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16,12.0,Low
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19,4.0,Low
3,TXN_7034554,Salad,2.0,5.0,10.0,Digital Wallet,Takeaway,2023-04-27,10.0,Low
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11,4.0,Low


## Reto final

Ahora que la base está limpia, responde estas preguntas utilizando código.

### 1. ¿Cuál es el producto con mayor cantidad vendida?

### 2. ¿Qué producto genera mayor revenue?


**Pista:** Utiliza la herramienta: `groupby()`

In [102]:
print(df.columns.tolist())

['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date', 'Revenue_Calculated', 'Revenue_Level']


In [ ]:
# ¿Cual es el producto con mayor catidad vendida?
item_mas_vendido = df.groupby("Item")["Quantity"].sum().idxmax()
print("El producto mas vendido es:", item_mas_vendido)


#¿Que producto genera mayor Revenue?
producto_mayor_revenue = df.groupby("Item")["Revenue_Calculated"].sum().idxmax()
print("El producto que genera mayor Revenue es:", producto_mayor_revenue)

El producto mas vendido es: Coffee
El producto que genera mayor Revenue es: Salad


In [104]:
item_mas_vendido2 = df.groupby("Item")["Quantity"].sum().sort_values(ascending=False).head(1)
print(item_mas_vendido2)

Item
Coffee    3551.0
Name: Quantity, dtype: float64
